# AI6130 Politeness Bias — Pilot Run & Diagnostics

## 1. Environment Initialization & Secure Secrets Loading

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

In [ ]:
# Mount Google Drive for real-time persistent storage of raw JSONL outputs
drive.mount("/content/drive")
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/AI6130 Group Project/results/raw")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Load tokens securely from Colab Secrets into environment variables
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GITHUB_PAT"] = userdata.get("GITHUB_PAT")

# Suppress Hugging Face download progress bars to prevent spamming blank lines/carriage returns in Colab cell output
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

## 2. Repository Sync & Colab Native Setup

In [ ]:
GITHUB_USER = "HJunyuan"
REPO_NAME = "ai6130-group-9-project"
BRANCH_NAME = "run/open-weight-pipeline"

In [ ]:
%cd /content
if not Path(f"/content/{REPO_NAME}").exists():
    !git clone -b {BRANCH_NAME} https://{os.environ["GITHUB_PAT"]}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd /content/{REPO_NAME}

!git pull origin {BRANCH_NAME}

In [ ]:
%%capture
!pip install -q uv
!uv pip install --system -e .

## 3. Prompt Integrity & Tokenizer Validation

In [ ]:
!python src/ai6130_group_9_project/prompts/validate_prompts.py \
    --input data/prompts_v1.jsonl \
    --templates data/templates.yaml

## 4. Execute Pilot Inference Pipeline

In [ ]:
import shutil

In [ ]:
PILOT_MODEL = "Qwen/Qwen3-32B"
PILOT_OUT_DRIVE = DRIVE_BACKUP_DIR / "pilot_Qwen--Qwen3-32B.jsonl"
PILOT_OUT_LOCAL = Path("results/raw") / PILOT_OUT_DRIVE.name
PILOT_OUT_LOCAL.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
%run src/ai6130_group_9_project/run/generate.py \
    --model "{PILOT_MODEL}" \
    --input data/prompts_pilot.jsonl \
    --batch-size 8 \
    --output "{PILOT_OUT_DRIVE}"

In [ ]:
# Keep a local mirror inside the repo directory for inspection
shutil.copy2(PILOT_OUT_DRIVE, PILOT_OUT_LOCAL)
print(f"Pilot output saved to Google Drive: {PILOT_OUT_DRIVE}")

## 5. Pilot Diagnostics & Ceiling Effect Analysis

In [ ]:
import re
import json
import pandas as pd

In [ ]:
meta = {}
with open("data/prompts_pilot.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        meta[row["prompt_id"]] = row

records = []
with open(PILOT_OUT_DRIVE, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        m = meta[r["prompt_id"]]
        text = r["response_text"]
        has_answer_tag = "Answer:" in text

        pred = None
        if has_answer_tag:
            after = text.split("Answer:")[-1].strip()
            if m["task"] == "gsm8k":
                nums = re.findall(r"-?\d+(?:\.\d+)?", after.replace(",", ""))
                pred = nums[0] if nums else None
            elif m["task"] != "xstest_safe":
                lets = re.findall(r"\b([A-E])\b", after)
                pred = lets[0] if lets else None

        correct = (pred == m["gold"]) if m["gold"] is not None else None
        logits = r.get("logits_meta") or {}

        records.append({
            "prompt_id": r["prompt_id"],
            "task": m["task"],
            "tone": m["tone"],
            "has_answer_tag": has_answer_tag,
            "correct": correct,
            "n_output_tokens": r["n_output_tokens"],
            "refusal_prob": logits.get("first_step_refusal_prob", 0.0),
        })

df = pd.DataFrame(records)

In [ ]:
print("=== 1. ACCURACY & CoT LENGTH BY TONE (Graded Tasks, n=80) ===")
graded_df = df[df["task"] != "xstest_safe"]
tone_summary = graded_df.groupby("tone").agg(
    format_compliance=("has_answer_tag", "mean"),
    accuracy=("correct", "mean"),
    mean_output_tokens=("n_output_tokens", "mean"),
).reindex(["very_rude", "blunt", "neutral", "polite", "very_deferential"])
display(tone_summary)

print("\n=== 2. CEILING EFFECT CHECK: ACCURACY BY TASK (Across Tones) ===")
task_summary = graded_df.groupby(["task", "tone"])["correct"].mean().unstack()
display(task_summary[["very_rude", "blunt", "neutral", "polite", "very_deferential"]])

print("\n=== 3. XSTEST SAFE REFUSAL PROPENSITY BY TONE (n=20) ===")
xst_df = df[df["task"] == "xstest_safe"]
xst_summary = xst_df.groupby("tone").agg(
    mean_first_step_refusal_prob=("refusal_prob", "mean"),
    max_first_step_refusal_prob=("refusal_prob", "max"),
    mean_output_tokens=("n_output_tokens", "mean"),
).reindex(["very_rude", "blunt", "neutral", "polite", "very_deferential"])
display(xst_summary)